#### AIRKOREA 2025/12/15 ~ 2025/12/24 관측 외 지역의 필터링 수행  

수행 방법: 2025/12/26의 관측소 지역 정보를 해당 기간에 동일 적용

In [5]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


def collect_valid_geoids(base_dir: str, ref_date: str) -> set:
    """
    기준 날짜(ref_date)의 모든 시간별 csv에서 값이 존재하는 geoId 집합 수집
    ref_date 형식: YYYY-MM-DD
    """
    dt = datetime.strptime(ref_date, "%Y-%m-%d")
    year, month, day = str(dt.year), str(dt.month), str(dt.day)

    ref_dir = os.path.join(base_dir, year, month, day)
    pattern = os.path.join(ref_dir, "*.csv")
    files = sorted(glob.glob(pattern))

    if not files:
        raise FileNotFoundError(f"기준 날짜 파일이 없습니다: {pattern}")

    valid_geoids = set()

    for file in files:
        df = pd.read_csv(file)

        if "geoId" not in df.columns or "stationName" not in df.columns:
            continue

        df["geoId"] = pd.to_numeric(df["geoId"], errors="coerce").astype("Int64")
        station_mask = (
            df["stationName"].notna() &
            (df["stationName"].astype(str).str.strip() != "") &
            (df["stationName"].astype(str).str.lower() != "nan")
        )

        valid_geoids.update(df.loc[station_mask, "geoId"].dropna().astype(int).tolist())

    return valid_geoids

def mask_airkorea_values_by_geoids(
    base_dir: str,
    start_date: str,
    end_date: str,
    valid_geoids: set,
    output_dir: str = None,
    overwrite: bool = False,
    keep_cols: list = None
):
    """
    start_date~end_date 기간의 AIRKOREA csv에서
    valid_geoids에 없는 geoId는 '값 컬럼만 NaN 처리'

    keep_cols:
      NaN 처리하지 않고 유지할 컬럼 목록
    """
    if output_dir is None and not overwrite:
        raise ValueError("output_dir를 지정하거나 overwrite=True로 설정해야 합니다.")

    if keep_cols is None:
        keep_cols = ["geoId", "geo_lon", "geo_lat", "dateTime"]

    current = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")

    total_files = 0
    total_rows = 0
    total_masked_rows = 0

    while current <= end_dt:
        year, month, day = str(current.year), str(current.month), str(current.day)
        day_dir = os.path.join(base_dir, year, month, day)
        pattern = os.path.join(day_dir, "*.csv")
        files = sorted(glob.glob(pattern))

        for file in files:
            df = pd.read_csv(file)

            if "geoId" not in df.columns:
                print(f"  - geoId 컬럼 없음, 스킵: {file}")
                continue

            df["geoId"] = pd.to_numeric(df["geoId"], errors="coerce").astype("Int64")

            # 유지할 컬럼 제외하고 NaN 처리 대상 컬럼 선택
            value_cols = [col for col in df.columns if col not in keep_cols]

            # 기준일 geoId에 없는 행
            invalid_mask = ~df["geoId"].isin(valid_geoids)

            before_masked = invalid_mask.sum()

            if value_cols:
                df.loc[invalid_mask, value_cols] = np.nan

            total_files += 1
            total_rows += len(df)
            total_masked_rows += before_masked

            if overwrite:
                save_path = file
            else:
                rel_path = os.path.relpath(file, base_dir)
                save_path = os.path.join(output_dir, rel_path)
                os.makedirs(os.path.dirname(save_path), exist_ok=True)

            df.to_csv(save_path, index=False)

            print(
                f"  - {os.path.basename(file)} | 전체 {len(df):,}행 | NaN 처리 대상 행 {before_masked:,}"
            )

        current += timedelta(days=1)

    print("\n작업 완료")
    print(f"총 파일 수: {total_files}")
    print(f"전체 행 수: {total_rows:,}")
    print(f"값 NaN 처리된 행 수 합계: {total_masked_rows:,}")

In [12]:
# AIRKOREA 최상위 폴더
base_dir = r"../../realtime_API/AIR_KOREA"

# 기준일: 이 날짜에 존재하는 geoId만 정상 관측 지점으로 간주
ref_date = "2025-12-26"

# 보정 대상 기간
start_date = "2025-12-15"
end_date = "2025-12-24"

# 새 폴더 저장
output_dir = r"../../realtime_API/AIR_KOREA"

valid_geoids = collect_valid_geoids(base_dir, ref_date)
print(f"기준 geoId 개수: {len(valid_geoids):,}")

C:\Users\USER\AppData\Local\Temp\ipykernel_34500\1986249359.py:26: DtypeWarning: Columns (0: so2Value, 1: coValue, 2: o3Value, 3: no2Value, 4: pm10Value, 5: pm25Value, 6: khaiValue, 7: so2Flag, 8: coFlag, 9: o3Flag, 10: no2Flag, 11: pm10Flag, 12: pm25Flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
C:\Users\USER\AppData\Local\Temp\ipykernel_34500\1986249359.py:26: DtypeWarning: Columns (0: so2Value, 1: pm10Value, 2: so2Flag, 3: pm10Flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
C:\Users\USER\AppData\Local\Temp\ipykernel_34500\1986249359.py:26: DtypeWarning: Columns (0: so2Value, 1: o3Value, 2: no2Value, 3: pm10Value, 4: so2Flag, 5: o3Flag, 6: no2Flag, 7: pm10Flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
C:\Users\USER\AppData\Local\Temp\ipykernel_34500\1986249359.py:26: DtypeWarning: Columns (0: so2Value, 1: coValue,

기준 geoId 개수: 661


In [13]:
mask_airkorea_values_by_geoids(
        base_dir=base_dir,
        start_date=start_date,
        end_date=end_date,
        valid_geoids=valid_geoids,
        output_dir=output_dir,
        overwrite=True
    )

  - AIR_KOREA_2025-12-15_03.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_04.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_05.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_06.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_07.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_08.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_09.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_10.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_11.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_12.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_13.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_14.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_15.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_16.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-15_17.csv | 전체 107,100행 | 

C:\Users\USER\AppData\Local\Temp\ipykernel_34500\1986249359.py:78: DtypeWarning: Columns (0: so2Value, 1: coValue, 2: o3Value, 3: no2Value, 4: pm10Value, 5: pm25Value, 6: khaiValue, 7: so2Flag, 8: coFlag, 9: o3Flag, 10: no2Flag, 11: pm10Flag, 12: pm25Flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


  - AIR_KOREA_2025-12-18_19.csv | 전체 107,100행 | NaN 처리 대상 행 106,439


C:\Users\USER\AppData\Local\Temp\ipykernel_34500\1986249359.py:78: DtypeWarning: Columns (0: so2Value, 1: coValue, 2: o3Value, 3: no2Value, 4: pm10Value, 5: pm25Value, 6: khaiValue, 7: so2Flag, 8: coFlag, 9: o3Flag, 10: no2Flag, 11: pm10Flag, 12: pm25Flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


  - AIR_KOREA_2025-12-18_20.csv | 전체 107,100행 | NaN 처리 대상 행 106,439


...

  - AIR_KOREA_2025-12-18_21.csv | 전체 107,100행 | NaN 처리 대상 행 106,439


C:\Users\USER\AppData\Local\Temp\ipykernel_34500\1986249359.py:78: DtypeWarning: Columns (0: khaiValue) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


  - AIR_KOREA_2025-12-18_22.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-18_23.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_00.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_01.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_02.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_03.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_04.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_05.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_06.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_07.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_08.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_09.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_10.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_11.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-19_12.csv | 전체 107,100행 | 

C:\Users\USER\AppData\Local\Temp\ipykernel_34500\1986249359.py:78: DtypeWarning: Columns (0: pm25Value, 1: pm25Flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


  - AIR_KOREA_2025-12-22_03.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_04.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_05.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_06.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_07.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_08.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_09.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_10.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_11.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_12.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_13.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_14.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_15.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_16.csv | 전체 107,100행 | NaN 처리 대상 행 106,439
  - AIR_KOREA_2025-12-22_17.csv | 전체 107,100행 | 